### LangSmith `evaluate()`로 모델 비교 (Upstage vs Gemini)

같은 질문 세트(데이터셋)를 **Upstage `solar-pro3`** 와 **Google `gemini-2.5-flash-lite`** 에 각각 던지고, 같은 평가 기준으로 점수를 매겨 비교합니다.

**흐름**: 데이터셋 생성 → 모델별 대상 함수(target) → 평가 함수(evaluators) → `evaluate()` 실행 → 결과 비교

| 용어 | 설명 |
|---|---|
| 데이터셋(Dataset) | 평가용 입력(질문) 모음. LangSmith 계정에 저장되어 재사용 가능 |
| 대상 함수(target) | 입력을 받아 모델 답변을 만드는 함수 (평가받는 대상) |
| 평가 함수(evaluator) | 답변을 보고 점수를 계산하는 함수 (이 노트북은 **코드 기반 규칙**만 사용) |
| 실험(Experiment) | 데이터셋 전체를 target 으로 한 번 실행하고 평가한 결과 묶음 |

> **비용 안내**: 질문 5개 × 모델 2개 = LLM 호출 10회가 발생하고, 데이터셋과 실험이 LangSmith 계정에 저장됩니다.

##### 1) 라이브러리 설치

In [ ]:
# uv add langsmith langchain-upstage langchain-google-genai

##### 2) 환경 변수 설정

`.env` 파일에 아래 값이 필요합니다. 값은 출력하지 않고 설정 여부만 확인합니다.
- `UPSTAGE_API_KEY`, `GOOGLE_API_KEY`
- `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`, `LANGSMITH_TRACING=true`

In [ ]:
import os
from dotenv import load_dotenv

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

REQUIRED_ENV = ["UPSTAGE_API_KEY", "GOOGLE_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f".env 파일에 다음 환경 변수가 없습니다: {', '.join(missing)}")

print("LangSmith 트레이싱:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

##### 3) 데이터셋 만들기

질문 5개를 LangSmith 데이터셋으로 등록합니다.
- 같은 이름의 데이터셋이 **이미 있으면 새로 만들지 않고 재사용**합니다. (셀을 다시 실행해도 중복 생성되지 않음)
- 이 데이터셋은 정답(reference)이 없고 질문(`inputs`)만 있습니다. 정답이 없으므로 평가는 답변 자체의 형식을 보는 규칙으로 합니다.
- 사이트에서는 **Datasets & Experiments** 메뉴에서 확인할 수 있습니다.

In [ ]:
from langsmith import Client

client = Client()

DATASET_NAME = "model-compare-qa"
QUESTIONS = [
    "LangChain과 LangGraph의 차이점은 무엇인가요?",
    "RAG(검색 증강 생성)를 초등학생도 이해할 수 있게 설명해 주세요.",
    "파이썬에서 리스트와 튜플의 차이를 알려 주세요.",
    "프롬프트 엔지니어링에서 few-shot 기법이란 무엇인가요?",
    "LangSmith를 사용하면 어떤 점이 좋은가요?",
]

if client.has_dataset(dataset_name=DATASET_NAME):
    print(f"데이터셋 '{DATASET_NAME}' 이(가) 이미 있어 재사용합니다.")
else:
    dataset = client.create_dataset(DATASET_NAME, description="Upstage vs Gemini 모델 비교용 질문")
    client.create_examples(
        dataset_id=dataset.id,
        examples=[{"inputs": {"question": q}} for q in QUESTIONS],
    )
    print(f"데이터셋 '{DATASET_NAME}' 생성 완료 ({len(QUESTIONS)}개 질문)")

##### 4) 대상 함수(target) 만들기

두 모델이 **같은 프롬프트**를 쓰도록 하고, 모델만 바꿉니다. 비교 실험에서는 모델 외의 조건(프롬프트, temperature)을 같게 맞추는 것이 중요합니다.

- `target(inputs)`: 데이터셋의 `inputs`(딕셔너리)를 받아 `{"answer": ...}` 딕셔너리를 반환
- 프롬프트에 "한국어로만, 300자 이내"를 명시했으므로, 모델이 이 지시를 지키는지가 평가 대상이 됩니다.
- `to_text()`: 모델에 따라 `content`가 문자열이 아니라 리스트일 수 있어 문자열로 통일

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_upstage import ChatUpstage
from langchain_google_genai import ChatGoogleGenerativeAI

MAX_ANSWER_CHARS = 300  # 프롬프트와 평가 기준에 함께 쓰는 답변 최대 글자 수

# 비교할 모델 (이름: 모델 객체). temperature 는 동일하게 0
MODELS = {
    "upstage-solar-pro3": ChatUpstage(
        model="solar-pro3", base_url="https://api.upstage.ai/v1", temperature=0
    ),
    "gemini-2.5-flash-lite": ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0),
}

prompt = ChatPromptTemplate.from_messages([
    ("system", f"당신은 유용한 AI 비서입니다. 반드시 한국어로만, {MAX_ANSWER_CHARS}자 이내로 답변하세요."),
    ("human", "{question}"),
])


def to_text(content) -> str:
    """모델 응답의 content 를 문자열로 변환합니다. (리스트면 텍스트 조각만 이어 붙임)"""
    if isinstance(content, str):
        return content
    return "".join(p if isinstance(p, str) else p.get("text", "") for p in content)


def make_target(llm):
    """주어진 모델로 질문에 답하는 target 함수를 만들어 반환합니다."""
    chain = prompt | llm

    def target(inputs: dict) -> dict:
        return {"answer": to_text(chain.invoke({"question": inputs["question"]}).content)}

    return target

##### 5) 평가 함수(evaluator) 만들기

정답과 비교하지 않고 **답변이 지시를 지켰는지**를 코드 규칙으로 점수화합니다. 평가 함수는 `outputs`(target 의 반환값)를 받아 `{"key": 이름, "score": 점수}`를 반환합니다.

| 평가 키 | 규칙 | 점수 | 점수의 의미 |
|---|---|---|---|
| `korean_ratio` | 답변의 글자(한글+영문) 중 한글 비율 | 0.0 ~ 1.0 | 1.0에 가까울수록 한국어 위주, 0에 가까울수록 영어 위주 (예: 0.92 = 한글 92%) |
| `no_english_preamble` | 답변 **앞 100자**의 한글 비율이 30% 이상 (영어 추론으로 시작하지 않음) | 0 또는 1 | **1 = 합격**(한국어로 시작), **0 = 불합격**(영어 추론으로 시작) |
| `length_ok` | 답변 길이가 300자 이하 (프롬프트의 길이 지시 준수) | 0 또는 1 | **1 = 합격**(1~300자), **0 = 불합격**(300자 초과 또는 빈 답변) |

**실험 결과를 읽는 방법**: 질문별 점수를 모델마다 평균 낸 값이 결과 표에 나옵니다. `no_english_preamble`과 `length_ok`는 0/1이므로 **평균이 곧 합격 비율**입니다. (예: 0.8 = 질문 5개 중 4개 합격)

> 점수는 **형식 기준**일 뿐 답변 내용이 맞는지는 판단하지 않습니다. 코드나 영어 용어가 많은 정상 답변은 `korean_ratio`가 낮게 나올 수 있으니, 점수가 낮은 답변은 실제 내용을 확인하세요.

> 앞에서 본 `solar-pro3`의 영어 추론 출력("The user asks...")을 `no_english_preamble`이 잡아내는지 확인하는 것이 핵심입니다.
> 평가 함수를 실험에 쓰기 전에, 아래에서 **가짜 답변으로 먼저 점검**합니다. (API 호출 없음)

In [ ]:
import re

KOREAN_RATIO_MIN = 0.3  # 앞부분이 한국어라고 인정하는 최소 한글 비율
PREAMBLE_CHARS = 100    # 앞부분으로 검사할 글자 수


def hangul_ratio(text: str) -> float:
    """한글과 영문 글자 중 한글의 비율을 반환합니다. (글자가 없으면 0.0)"""
    ko = len(re.findall(r"[가-힣]", text))
    en = len(re.findall(r"[A-Za-z]", text))
    return ko / (ko + en) if ko + en else 0.0


def korean_ratio(outputs: dict) -> dict:
    return {"key": "korean_ratio", "score": hangul_ratio(outputs["answer"])}


def no_english_preamble(outputs: dict) -> dict:
    ok = hangul_ratio(outputs["answer"][:PREAMBLE_CHARS]) >= KOREAN_RATIO_MIN
    return {"key": "no_english_preamble", "score": int(ok)}


def length_ok(outputs: dict) -> dict:
    return {"key": "length_ok", "score": int(0 < len(outputs["answer"]) <= MAX_ANSWER_CHARS)}


EVALUATORS = [korean_ratio, no_english_preamble, length_ok]

# 평가 함수 점검: 가짜 답변으로 점수가 의도대로 나오는지 확인 (API 호출 없음)
samples = {
    "정상 한국어 답변": "랭체인은 LLM 앱 개발 프레임워크이고, 랭그래프는 상태 기반 워크플로우 라이브러리입니다.",
    "영어 추론으로 시작": "The user asks in Korean about LangChain. We need to answer in Korean. 랭체인은 프레임워크입니다.",
    "너무 긴 답변": "가" * 400,
}
for label, text in samples.items():
    scores = {e.__name__: round(e({"answer": text})["score"], 2) for e in EVALUATORS}
    print(f"{label:<20} {scores}")

##### 6) `evaluate()` 실행

모델마다 `evaluate()`를 한 번씩 호출해 **실험 2개**를 만듭니다.

| 파라미터 | 설명 |
|---|---|
| 첫 번째 인자 | 평가 대상 함수 (target) |
| `data` | 데이터셋 이름 (질문 5개가 하나씩 target 에 전달됨) |
| `evaluators` | 답변마다 실행할 평가 함수 목록 |
| `experiment_prefix` | 실험 이름 앞부분. 뒤에 임의 문자열이 붙어 사이트에서 모델별로 구분됨 |
| `metadata` | 실험에 붙일 부가 정보 (모델명) |
| `max_concurrency` | 동시에 실행할 최대 개수. 낮추면 속도 제한(rate limit)을 피하기 쉬움 |

> Upstage 호출 5회 + Gemini 호출 5회가 발생합니다. 실행에 1~2분 걸릴 수 있습니다.

In [ ]:
from langsmith import evaluate

MAX_CONCURRENCY = 2  # 동시 호출 수 (속도 제한 방지를 위해 낮게 설정)

results = {}
for name, llm in MODELS.items():
    print(f"\n▶ {name} 실험 실행")
    results[name] = evaluate(
        make_target(llm),
        data=DATASET_NAME,
        evaluators=EVALUATORS,
        experiment_prefix=name,
        metadata={"model": name},
        max_concurrency=MAX_CONCURRENCY,
    )

##### 7) 결과 비교

**코드에서 확인**: 실험 결과를 순회하며 평가 키별 평균 점수를 계산해 두 모델을 나란히 출력합니다.

**LangSmith 사이트에서 확인**
1. **Datasets & Experiments** 메뉴에서 `model-compare-qa` 데이터셋을 선택
2. **Experiments** 탭에서 두 실험(`upstage-solar-pro3-...`, `gemini-2.5-flash-lite-...`)을 체크
3. **Compare** 버튼을 누르면 질문별 답변과 평가 점수를 나란히 비교
4. 점수가 낮은 질문을 클릭해 **실제 답변과 트레이스**를 확인 (예: 영어 추론으로 시작한 답변)

> 질문이 5개뿐이고 실행마다 답변이 조금씩 달라질 수 있으므로, 이 결과는 경향을 보는 용도입니다.
> 사이트 메뉴 이름은 화면과 다를 수 있습니다.

In [ ]:
from collections import defaultdict

def average_scores(experiment_results) -> dict:
    """실험 결과를 순회하며 평가 키별 평균 점수를 계산합니다."""
    scores = defaultdict(list)
    for row in experiment_results:
        for r in row["evaluation_results"]["results"]:
            if r.score is not None:
                scores[r.key].append(r.score)
    return {key: sum(v) / len(v) for key, v in scores.items()}


# 실험 이름 출력: LangSmith 사이트 Experiments 목록의 이름과 대조하면 같은 실행인지 확인할 수 있음
print("■ 실험 이름 (사이트의 Experiment 열과 대조)")
for name, res in results.items():
    print(f"   {name:<24} → {res.experiment_name}")

averages = {name: average_scores(res) for name, res in results.items()}
keys = [e.__name__ for e in EVALUATORS]

print("\n■ 평가 키별 평균 점수")
print(f"{'평가 키':<22}" + "".join(f"{name:>24}" for name in averages))
for key in keys:
    print(f"{key:<22}" + "".join(f"{avg.get(key, float('nan')):>24.2f}" for avg in averages.values()))

##### 8) 결과 분석

LangSmith 사이트의 Experiments 화면(`model-compare-qa`, 질문 5개, 반복 1회, 오류율 0%)을 기준으로 정리했습니다. 두 실험은 같은 데이터셋 버전으로 실행되었습니다. 그래프에서 읽은 값은 **대략적인 값**입니다.

| 지표 | upstage-solar-pro3 | gemini-2.5-flash-lite |
|---|---|---|
| `korean_ratio` (평균) | 0.86 | 0.86 |
| `no_english_preamble` (평균) | 1.00 | 1.00 |
| `length_ok` (평균) | **1.00** | **0.00** |
| 지연 시간 P50 / P99 | 0.52초 / 0.69초 | 6.36초 / 7.17초 |
| 입력 토큰 (Tokens 그래프) | 약 510 | 약 220 |
| 출력 토큰 (Tokens 그래프) | 약 360 | 약 830 |
| 비용 (Cost 그래프, LangSmith 추정치) | 약 $0.00022 | 약 $0.00036 |

**1. 가장 큰 차이는 길이 지시 준수(`length_ok`)**
- Upstage는 5개 질문 모두 300자 이내로 답했고, Gemini는 5개 모두 300자를 넘겼습니다.
- 두 모델이 받은 프롬프트("한국어로만, 300자 이내")는 같으므로 **모델이 길이 지시를 지키는 정도가 다르다**고 볼 수 있습니다.
- LLM은 글자 수를 정확히 세지 못해 "N자 이내" 같은 수치 제약이 잘 안 지켜지는 경향이 있습니다. 이 실험은 그 경향이 모델마다 다르게 나타날 수 있음을 보여 줍니다.
- 길이가 중요하다면 프롬프트를 더 구체적으로(예: "3문장 이내") 바꾸거나, `max_output_tokens`로 제한하거나, 코드에서 후처리하는 방법이 있습니다. 토큰 제한은 문장 중간에서 잘릴 수 있어 주의가 필요합니다.

**2. 토큰: 입력은 Upstage가 많고, 출력은 Gemini가 많음**
- **출력 토큰**: Gemini가 Upstage의 약 2배 이상입니다. 길이 지시를 어긴 결과(`length_ok = 0`)와 같은 방향입니다.
- **입력 토큰**: 같은 프롬프트와 질문인데 Upstage가 약 2배 이상 많습니다. 모델마다 토크나이저(tokenizer)가 달라 같은 한국어 문장도 토큰 수가 다르게 계산되는 것으로 보입니다. 원인을 직접 확인하지는 못했습니다.
- 그래프 수치가 5회 합계인지 평균인지는 화면에서 확인이 필요합니다. 두 모델의 **상대 비교**에는 영향이 없습니다.

**3. 지연 시간: 답변 길이만으로는 설명되지 않음**
- Gemini의 P50이 Upstage의 약 12배인데, 출력 토큰은 약 2배 수준입니다. 길이 차이가 한 원인일 수는 있지만 **지연 시간 차이 전체를 설명하지는 못합니다.**
- 모델 처리 속도, 서버 상태, 네트워크 경로 등 다른 요인이 있을 수 있으나 이 화면만으로는 원인을 알 수 없습니다. 실행 로그에서도 Upstage는 5건에 약 2초, Gemini는 약 16초가 걸렸습니다.

**4. 비용: Gemini가 더 높은 것으로 추정, 단 근거가 약함**
- LangSmith의 추정치 기준으로 Gemini가 Upstage의 약 1.6배입니다. Gemini는 입력이 적지만 출력 토큰이 많아 **출력 비용이 대부분**을 차지합니다.
- 비용은 LangSmith가 모델별 단가표로 계산한 **추정치**입니다. 특히 Upstage 모델의 단가가 정확히 반영되었는지는 확인하지 못했으니, 실제 비용은 각 서비스의 콘솔(사용량 화면)에서 확인하세요.
- 절대 금액이 매우 작아(5건에 1만분의 몇 달러) 이 실험만으로 모델 선택을 결정할 수준은 아닙니다. 운영에서는 호출 횟수에 비례해 커집니다.

**5. `korean_ratio`가 1.0이 아닌 것은 문제가 아님**
- 두 모델 모두 0.86입니다. 질문에 `LangChain`, `RAG`, `few-shot` 같은 영어 용어가 들어 있어 답변에도 영문 글자가 섞이기 때문입니다. 두 모델이 서로 비슷하다는 것이 의미 있는 결과입니다.

**6. `no_english_preamble`이 모두 1.00**
- 앞에서 본 `solar-pro3`의 영어 추론 출력("The user asks...")이 이번 실험에서는 나타나지 않았습니다.
- 이번 프롬프트에는 "반드시 한국어로만"이 들어 있어서 그 영향일 수 있습니다. 다만 앞의 예제와 프롬프트 외 조건도 다르므로 **원인을 단정할 수는 없습니다.**

**7. 종합**
- 이 프롬프트와 질문 5개 기준으로는 **Upstage `solar-pro3`가 길이 준수, 지연 시간, 추정 비용에서 앞섰습니다.** 한국어 비율과 영어 추론 여부는 차이가 없었습니다.
- 다만 아래 한계 때문에 "어느 모델이 더 좋다"는 결론이 아니라 **이 조건에서 나타난 경향**으로 보셔야 합니다.

**8. 이 결과를 해석할 때의 한계**
- **표본이 작음**: 질문 5개, 반복 1회라 경향만 볼 수 있습니다. 같은 실험을 `num_repetitions=3`으로 반복하면 실행 간 변동을 확인할 수 있습니다.
- **형식 기준만 평가**: 답변 내용이 정확한지는 판단하지 않습니다. "길이를 지켰지만 내용이 틀린 답변"은 높은 점수를 받습니다. 내용 평가가 필요하면 별도의 평가 방식이 필요합니다.
- **노트북 표와 사이트 값의 차이**: 노트북 결과 표에서 Gemini의 `korean_ratio`가 0.89, `length_ok`가 0.20으로 출력된 적이 있어 사이트의 0.86, 0.00과 달랐습니다. 두 실험은 실행 로그의 이름이 사이트와 같아 같은 실험입니다. 결과 셀(`cell-14`)을 다시 실행해(API 호출 없이 계산만 함) 값을 대조하세요.

**다음에 해볼 것**
1. 사이트의 Compare 화면에서 Gemini의 `length_ok = 0` 답변을 열어 실제 글자 수와 출력 토큰 수를 확인
2. 프롬프트의 길이 지시를 바꾸고 같은 데이터셋으로 실험을 다시 실행해 `length_ok`와 출력 토큰 변화를 확인
3. `evaluate(..., num_repetitions=3)`으로 반복 실행하여 결과의 안정성 확인